In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [ ]:
deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
len(deals)

In [ ]:
deals[10].describe()

### Prompting GPT-5-mini to summarize deals and identify their price

In [ ]:
SYSTEM_PROMPT = """From the provided list, select the 5 deals with the most detailed, high-quality product descriptions and a clearly stated final price.

Return only valid JSON (no extra text) using the specified format.
Include the price as a numeric value extracted from the description.

Exclude any deal where the price is unclear or only expressed as a discount (e.g., “$100 off,” “reduced by $50”).

Prioritize depth and clarity of the product description over deal terms or conditions.
Only include deals where you are highly confident in the actual product price. 
"""

USER_PROMPT_PREFIX = """Select the 5 most promising deals from the list based on the quality and detail of the product description and the presence of a clear, non-zero final price.

Rewrite each selected description as a concise summary of the product itself, excluding any deal terms or promotional language.

For each of the 5 items, populate the product_description field with a short paragraph.

Ignore entries where the price is unclear or only presented as a discount (e.g., “$200 off,” “save $50”). Only include products when you are highly confident in the actual price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# this makes a required user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
# A user prompt for the deals scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

In [ ]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("Samsung 60-inch LED TV available at an excellent price", 300, 1000, "www.samsung.com")